# Xử lý Ngôn ngữ Tự nhiên (NLP) với Bi-LSTM
Tập dữ liệu được sử dụng là **IMDB Movie Reviews** chứa 50.000 bài đánh giá phim đã được gán nhãn (tích cực/tiêu cực).

## 1. Cơ sở lý thuyết nhanh
* **RNN (Recurrent Neural Network):** Mạng nơ-ron chuyên xử lý dữ liệu chuỗi (sequence) như văn bản, âm thanh. Tuy nhiên, RNN cơ bản gặp vấn đề "Vanishing Gradient" (Tiêu biến đạo hàm), khiến nó không thể "nhớ" được các thông tin ở xa trong quá khứ đối với các câu văn dài.
* **LSTM (Long Short-Term Memory):** Một bản nâng cấp của RNN, sử dụng hệ thống "Cổng" (Gates: Forget, Input, Output) và "Trạng thái tế bào" (Cell State) để quyết định thông tin nào cần giữ lại, thông tin nào nên quên đi. LSTM giải quyết cực kỳ tốt vấn đề ghi nhớ dài hạn.
* **Bidirectional LSTM (Bi-LSTM):** Kỹ thuật chạy hai lớp LSTM song song: một lớp đọc văn bản từ trái sang phải, một lớp đọc từ phải sang trái. Điều này giúp mô hình hiểu được ngữ cảnh của một từ dựa trên cả những từ đứng trước VÀ những từ đứng sau nó, mang lại hiệu suất vượt trội trong NLP.

## 2. Import các thư viện cần thiết

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Dropout, Bidirectional
from tensorflow.keras.callbacks import ReduceLROnPlateau, Callback

# Thiết lập phong cách cho các biểu đồ (plots)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# --- Tối ưu hóa bộ nhớ GPU ---
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("Đã bật chế độ GPU Memory Growth.")
    except RuntimeError as e:
        print(e)
else:
    print("Không tìm thấy GPU, sử dụng CPU.")

## 3. Tải và Khám phá dữ liệu (Exploratory Data Analysis - EDA)
Việc hiểu dữ liệu trước khi đưa vào mô hình là bắt buộc đối với một Data Scientist/Machine Learning Engineer.

In [ ]:
# Đường dẫn tới dataset trên Kaggle. Bạn có thể thay đổi đường dẫn nếu chạy trên máy cá nhân.
file_path = "/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv"

# Fallback để có thể chạy thử nghiệm nếu không tìm thấy file
if not os.path.exists(file_path):
    file_path = "IMDB Dataset.csv" # Giả sử file nằm ở thư mục hiện tại

try:
    df = pd.read_csv(file_path)
    print(f"✓ Dữ liệu tải thành công: Lấy được {df.shape[0]} bài đánh giá và {df.shape[1]} cột.")
    display(df.head())
except FileNotFoundError:
    print("✗ Lỗi: Không tìm thấy file dữ liệu. Vui lòng kiểm tra lại đường dẫn.")

### 3.1. Phân bố nhãn (Label Distribution)
Kiểm tra xem tập dữ liệu có bị mất cân bằng (Imbalanced) hay không. Một tập dữ liệu cân bằng giúp mô hình không bị "thiên vị" về một class cụ thể nào.

In [ ]:
# Đếm số lượng các nhãn
label_counts = df['sentiment'].value_counts()
print(label_counts)

# Vẽ biểu đồ Bar Chart cho phân bố nhãn
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='sentiment', palette='viridis')
plt.title('Phân bố Cảm xúc (Sentiment Distribution)', fontsize=14, fontweight='bold')
plt.xlabel('Cảm xúc')
plt.ylabel('Số lượng')
plt.show()

### 3.2. Phân bố độ dài bài đánh giá (Review Lengths)
Mô hình yêu cầu đầu vào có độ dài cố định. Việc vẽ biểu đồ phân bố số lượng từ trong các bài review sẽ giúp chúng ta quyết định được tham số `MAX_LENGTH` (Độ dài tối đa của câu) một cách khoa học nhất, thay vì chọn bừa.

In [ ]:
# Tính số lượng từ trong mỗi bài đánh giá
review_lengths = df['review'].apply(lambda x: len(str(x).split()))

# Vẽ biểu đồ Histogram
plt.figure(figsize=(10, 6))
sns.histplot(review_lengths, bins=50, kde=True, color='blue')
plt.title('Phân bố độ dài bài đánh giá (Số từ)', fontsize=14, fontweight='bold')
plt.xlabel('Số từ trong một Review')
plt.ylabel('Tần suất')

# Hiển thị các đường đánh dấu các phân vị (percentiles)
plt.axvline(np.percentile(review_lengths, 50), color='r', linestyle='--', label='Trung vị (Median)')
plt.axvline(np.percentile(review_lengths, 90), color='g', linestyle='-.', label='Phân vị 90%')
plt.axvline(450, color='black', linestyle=':', label='MAX_LENGTH = 450 (Đề xuất)')
plt.legend()
plt.show()

print(f"Số từ trung bình: {np.mean(review_lengths):.0f} từ")
print(f"90% bài đánh giá có độ dài nhỏ hơn: {np.percentile(review_lengths, 90):.0f} từ")

Dựa vào biểu đồ trên, ta thấy việc giới hạn câu ở độ dài `MAX_LENGTH = 450` từ là hoàn toàn hợp lý, vì nó bao phủ được phần lớn dữ liệu mà không làm mô hình bị quá nặng (do padding các câu ngắn lên kích thước quá lớn).

## 4. Tiền xử lý dữ liệu (Data Preprocessing)

In [ ]:
# Tách Text và Labels
X = df["review"].astype(str)
y = df["sentiment"]

# Chia dữ liệu theo tỷ lệ: 70% Train, 15% Validation, 15% Test
X_train_raw, X_temp, y_train_raw, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val_raw, X_test_raw, y_val_raw, y_test_raw = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Kích thước tập Training: {len(X_train_raw)} mẫu")
print(f"Kích thước tập Validation: {len(X_val_raw)} mẫu")
print(f"Kích thước tập Testing: {len(X_test_raw)} mẫu")

# Chuẩn bị Label (Chuyển "positive"/"negative" thành 0, 1)
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
y_test = label_encoder.transform(y_test_raw)
num_classes = len(label_encoder.classes_)
print(f"Các nhãn (classes) được học: {label_encoder.classes_} -> {np.unique(y_train)}")

### Mã hóa văn bản (Tokenization & Padding)
Máy tính không hiểu được chữ, nên ta phải gán mỗi từ thành một con số nguyên (Tokenization). Sau đó, ta làm cho tất cả các câu có độ dài bằng nhau bằng cách thêm số 0 vào cuối các câu ngắn (Padding).

In [ ]:
MAX_LENGTH = 450

tokenizer = Tokenizer(oov_token="<OOV>") # OOV: Out-of-vocabulary (Xử lý các từ chưa từng gặp)
tokenizer.fit_on_texts(X_train_raw)
vocab_size = len(tokenizer.word_index) + 1

# Chuyển chữ thành số và đệm câu
X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_raw), maxlen=MAX_LENGTH, padding="post")
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(X_val_raw), maxlen=MAX_LENGTH, padding="post")
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test_raw), maxlen=MAX_LENGTH, padding="post")

print(f"Kích thước từ vựng (Vocabulary size): {vocab_size}")
print(f"Kích thước ma trận X_train_pad: {X_train_pad.shape}")

## 5. Xây dựng cấu trúc mô hình Bi-LSTM
Cấu trúc mô hình được xây dựng bao gồm:
1.  **Lớp Embedding:** Chuyển đổi các chỉ số nguyên từ (Token ID) thành các vector ngữ nghĩa (dense vectors).
2.  **Lớp SpatialDropout1D:** Một dạng dropout đặc biệt cho Embedding, giúp tránh Overfitting bằng cách loại bỏ ngẫu nhiên toàn bộ trục đặc trưng của vector.
3.  **Lớp Bidirectional(LSTM):** Trích xuất đặc trưng chuỗi theo 2 chiều ngữ cảnh.
4.  **Các lớp Dense:** Xử lý và phân loại đưa ra kết quả cuối cùng.

In [ ]:
EMBEDDING_DIM = 300

model = tf.keras.Sequential([
    # Lớp nhúng (Embedding Layer)
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LENGTH,
        trainable=True
    ),
    SpatialDropout1D(0.2),

    # Bi-LSTM Layer 1 (Return sequences = True để nối tiếp với LSTM phía sau)
    Bidirectional(LSTM(128, dropout=0.1, return_sequences=True)),

    # Bi-LSTM Layer 2
    Bidirectional(LSTM(64, dropout=0.1)),

    # Fully Connected Layers
    Dense(64, activation="relu"),
    Dropout(0.2),

    # Lớp Output (Sử dụng softmax cho num_classes)
    Dense(num_classes, activation="softmax")
])

# Biên dịch mô hình (Compile)
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-4),
    metrics=["accuracy"]
)

model.summary()

## 6. Huấn luyện mô hình
Chúng ta sẽ định nghĩa một **Custom Callback** để tính F1-Score trên tập Validation sau mỗi Epoch (vì metrics mặc định của Keras không hỗ trợ F1-Score đa nhãn tốt bằng sklearn).
Đồng thời kết hợp **ReduceLROnPlateau** để giảm Learning Rate tự động nếu mô hình ngừng học.

In [ ]:
# --- CUSTOM CALLBACK CHO F1-SCORE ---
class F1ScoreCallback(Callback):
    def __init__(self, validation_data):
        super().__init__()
        self.validation_data = validation_data

    def on_epoch_end(self, epoch, logs=None):
        X_val, y_val = self.validation_data
        y_pred_probs = self.model.predict(X_val, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        current_f1 = f1_score(y_val, y_pred, average='weighted')
        logs['val_f1'] = current_f1
        print(f" — val_f1_score: {current_f1:.4f}")

f1_callback = F1ScoreCallback(validation_data=(X_val_pad, y_val))
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, verbose=1, min_lr=1e-6)

print("--- BẮT ĐẦU HUẤN LUYỆN ---")
history = model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val_pad, y_val),
    callbacks=[reduce_lr, f1_callback],
    verbose=1
)

## 7. Trực quan hóa quá trình huấn luyện (Learning Curves)
Vẽ biểu đồ Loss và Accuracy qua từng Epoch giúp chúng ta nhận biết mô hình có bị Underfitting hay Overfitting hay không.

In [ ]:
# Lấy lịch sử từ callback
acc = history.history.get('accuracy', [])
val_acc = history.history.get('val_accuracy', [])
loss = history.history.get('loss', [])
val_loss = history.history.get('val_loss', [])

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14, 6))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy', marker='o')
if val_acc:
    plt.plot(epochs_range, val_acc, label='Validation Accuracy', marker='o')
plt.title('Training and Validation Accuracy', fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss', marker='o')
if val_loss:
    plt.plot(epochs_range, val_loss, label='Validation Loss', marker='o')
plt.title('Training and Validation Loss', fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## 8. Đánh giá chuyên sâu trên tập Test (Dữ liệu chưa từng thấy)
Dùng biểu đồ Confusion Matrix để xem chi tiết mô hình đoán đúng/sai ở những nhãn nào nhiều nhất, và in ra báo cáo Classification Report (Precision, Recall, F1-Score).

In [ ]:
print("--- ĐÁNH GIÁ TRÊN TẬP TEST ---")
test_loss, test_acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Độ chính xác (Test Accuracy): {test_acc*100:.2f}%\n")

# Dự đoán nhãn
y_pred_probs = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_probs, axis=1)

# 8.1 Vẽ Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix - Tập Test', fontsize=14, fontweight='bold')
plt.xlabel('Dự đoán (Predicted Label)')
plt.ylabel('Thực tế (True Label)')
plt.show()

# 8.2 In Classification Report
print("\n--- Báo cáo phân loại (Classification Report) ---")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))